# Experiment E — Bertrand pricing pilot (διακριτό grid)  
## ΠΡΟΣΟΧΗ: template — ενεργοποιείται μετά το group-level Game interface

**Τι θα ελέγξουμε:** το κεντρικό goal σε μικρή κλίμακα — το ίδιο κανάλι που φτιάχνει συνεργασία στο PD, φτιάχνει καρτέλ σε αγορά; 4 πωλητές, κοινή αγορά, τιμές {1..5}, κόστος 1, ο φθηνότερος παίρνει τη ζήτηση. Nash: τιμή≈κόστος. Καρτέλ: όλοι στο 5.

**Conditions:** no_comm vs baseline_cheap_talk (+framing_competitive αν υπάρχει quota). **Μετρικές:** μέση τιμή ανά γύρο, convergence round (όλες οι τιμές ≥4 για 3 συνεχόμενους), breakdown rate, ρητό price signaling στα μηνύματα.

**Σημείωση:** στα 7B μοντέλα ο pilot μετράει πρώτα αν το παιχνίδι διαβάζεται καθόλου — και αρνητικό αποτέλεσμα ορίζει το capability floor πριν τα μεγάλα μοντέλα.

## Setup — install, GPU check, clone, HF token

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (χρειάζεται για Llama/Gemma)

**Προσοχή:** το repo πρέπει να έχει γίνει push με τις αλλαγές του Phase 1.5 (topologies + `--action-retries`) πριν τρέξει αυτό το notebook.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo
# sanity: Phase-1.5 features present
assert 'clique' in open('topology.py').read(), 'Repo lacks Phase-1.5 topologies — push first!'
!ls

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print('No HF_TOKEN secret (fine for Qwen):', e)

In [ ]:
# Gate
import os
assert os.path.exists('games_market.py'), (
    'Το group-level Game interface (games_market.py) δεν έχει μπει ακόμα — επόμενο βήμα κώδικα.')

In [ ]:
# Σχεδιασμένη εντολή:
# !python run_bertrand.py --provider local --model-id Qwen/Qwen2.5-7B-Instruct \
#     --conditions no_comm cheap_talk --price-grid 1 2 3 4 5 --cost 1 \
#     --n-runs 5 --n-rounds 16